# 04 — Grid Search HDBSCAN y selección lexicográfica (Fase 1)

En esta primera fase buscamos, para cada una de las tres representaciones del texto de las earnings calls (TF-IDF, FinBERT y SBERT), la configuración de HDBSCAN que produce la partición más sólida bajo validación temporal *walk-forward*. El resultado es un único candidato congelado por representación, que pasa a la validación económica de la Fase 2.

**Idea central de la validación temporal.** Para cada año de corte $T$ ajustamos el pipeline (vectorizado TF-IDF, proyección UMAP y clustering HDBSCAN) sobre toda la historia acumulada hasta $T$ (ventana expansiva) y leemos las etiquetas de clúster sobre las observaciones del propio año $T$. El aislamiento temporal garantiza que ni el vocabulario de TF-IDF ni la geometría de UMAP llegan a ver años posteriores. Ahora bien, como las observaciones del año $T$ forman parte del conjunto de ajuste, las métricas de esta fase **no son una estimación fuera de muestra**: constituyen una **cota superior** de la calidad estructural alcanzable, es decir, cuán separables y coherentes son los grupos en el mejor caso, cuando el modelo ya ha visto esas observaciones. Esta fase sirve por tanto para descartar configuraciones geométricamente inviables y quedarnos con un finalista por representación. La evaluación predictiva genuina, sobre retornos de un año ajeno al entrenamiento, se reserva para la validación económica de la Fase 2 (cuaderno 05).

La salida del notebook es `Seleccion_Finalistas.csv`, con los tres candidatos que entran en la Fase 2.


## Bloque 1 — Configuración

Dejamos a la vista, al principio, los números que definen el experimento completo: los parámetros de TF-IDF y UMAP, el espacio de búsqueda de HDBSCAN y el año a partir del cual arranca la validación temporal. 

**Nota**: En este cuaderno llamamos *fold* a cada partición definida por un año $T$

In [1]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

import warnings

import umap.umap_ as umap
import hdbscan
from hdbscan.validity import validity_index
from scipy.sparse import issparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.cluster import normalized_mutual_info_score

In [2]:
# Raíz del proyecto. Si se ejecuta el notebook desde otra carpeta hay que cambiar esto.
PROJECT_ROOT = Path.cwd()

DATASET_LEMATIZADO = PROJECT_ROOT / "data" / "processed" / "Dataset_lematizado.parquet"
MATRIZ_FINBERT     = PROJECT_ROOT / "data" / "matrices" / "Matriz_FinBERT.npy"
MATRIZ_SBERT       = PROJECT_ROOT / "data" / "matrices" / "Matriz_SBERT.npy"

REGISTRY_PATH   = PROJECT_ROOT / "experiments" / "registry.csv"
FINALISTAS_PATH = PROJECT_ROOT / "experiments" / "Seleccion_Finalistas.csv"

REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)

In [3]:
# TF-IDF: el vectorizador se reajusta dentro de cada fold (ver Bloque 4), no aquí.
TFIDF_MAX_DF       = 0.85
TFIDF_MIN_DF       = 0.01
TFIDF_MAX_FEATURES = 5000

# UMAP. Misma semilla para todos los folds y representaciones; sin esto la
# proyeccion cambia entre ejecuciones y no podriamos comparar nada.
UMAP_PARAMS = {
    "n_neighbors":  15,
    "n_components": 50,
    "metric":       "cosine",
    "random_state": 42,
}

# Silhouette se calcula sobre la proyección UMAP, que ya vive en R^50 euclideo.
SILHOUETTE_METRIC = "euclidean"

In [4]:
# Rejilla de HDBSCAN: 6 x 4 x 4 x 2 = 192 combinaciones por representacion
# (576 en total entre las tres).
HDBSCAN_GRID = {
    "min_cluster_size":         [150, 200, 250, 300, 400, 500],
    "min_samples":              [5, 15, 20, 30],
    "cluster_selection_epsilon": [0.0, 0.1, 0.3, 0.5],
    "cluster_selection_method":  ["eom", "leaf"],
}

# TF-IDF va con matrix_key=None porque se vectoriza sobre la marcha;
# FinBERT y SBERT cargan su matriz densa ya calculada.
REPRESENTACIONES = [
    {"name": "tfidf",   "matrix_key": None},
    {"name": "finbert", "matrix_key": "finbert"},
    {"name": "sbert",   "matrix_key": "sbert"},
]

MATRIX_PATHS = {"finbert": MATRIZ_FINBERT, "sbert": MATRIZ_SBERT}

In [5]:
# El primer train termina en 2020. Cada fold ajusta el pipeline sobre el
# historico <= T y lee las etiquetas sobre el propio año T (cota superior).
TRAIN_MIN_END_YEAR = 2020

# Lo que guardamos de cada experimento para comparar configuraciones despues.
METRIC_KEYS = ["dbcv", "silhouette", "nmi", "n_clusters", "noise_ratio"]

## Bloque 2 — Cálculos auxiliares

Reunimos aquí solo las piezas de cálculo que se repiten en los bloques siguientes: la enumeración de las ventanas temporales, la carga de los embeddings con su comprobación de alineación y el cálculo de las métricas de calidad de un clustering. El recorrido experimental propiamente dicho aparece después.

### Ventanas walk-forward

Para cada año de corte $T$, desde `TRAIN_MIN_END_YEAR` hasta el penúltimo año disponible, el conjunto de ajuste son todos los trimestres con año $\le T$ y las métricas se leen sobre las observaciones del propio año $T$. La ventana es expansiva: a medida que avanza $T$ el ajuste acumula más historia, sin incorporar en ningún momento documentos de años posteriores a $T$. El último año del corpus no genera fold en esta fase y queda reservado para la validación económica de la Fase 2.


In [6]:
def generar_ventanas_walk_forward(quarters, train_min_end_year=TRAIN_MIN_END_YEAR):
    '''Ventanas expanding-window: para cada año de corte T, train con anio <= T.
    Las metricas de Fase 1 se leen sobre el propio año T (cota superior), por lo
    que la ventana no define un conjunto de validacion separado. El recorrido de T
    llega hasta el penultimo año disponible, de modo que el ultimo año del corpus
    queda integro para la validacion economica de la Fase 2.'''
    qs = sorted({str(q) for q in quarters})
    if not qs:
        return []

    anios = sorted({int(q[:4]) for q in qs})
    anio_min, anio_max = anios[0], anios[-1]

    ventanas = []
    fold = 1
    for train_end_year in range(train_min_end_year, anio_max):
        train_q = [q for q in qs if anio_min <= int(q[:4]) <= train_end_year]
        tiene_anio_T = any(int(q[:4]) == train_end_year for q in qs)
        if not train_q or not tiene_anio_T:
            continue
        ventanas.append({
            "fold": fold,
            "train_end_year": train_end_year,
            "train_quarters": train_q,
        })
        fold += 1
    return ventanas


### Carga de embeddings con verificación de alineación

FinBERT y SBERT se calcularon en un paso anterior y se guardaron como `.npy`. Necesitamos garantizar que la fila $i$ de la matriz se corresponde con la fila $i$ del dataset lematizado; de lo contrario estaríamos clusterizando embeddings emparejados con la empresa equivocada. Comprobamos primero que coinciden en número de filas y, si existe el fichero de identificadores paralelo, exigimos además igualdad estricta por `tupla_id` en el mismo orden.

In [7]:
def _verificar_alineacion_matriz(matriz_path, matriz, expected_ids, repr_name):
    '''Comprueba que la matriz y el dataset están fila a fila en el mismo orden.'''
    expected = np.asarray(expected_ids, dtype=object)
    n_mat = int(matriz.shape[0])
    n_exp = int(len(expected))
    if n_mat != n_exp:
        raise AssertionError(
            f"{repr_name}: la matriz tiene {n_mat} filas y el dataset {n_exp}."
        )

    ids_path = str(matriz_path)[:-4] + "_ids.npy"
    if not Path(ids_path).exists():
        print(f"    [aviso] {repr_name}: no encuentro {ids_path}; solo compruebo el número de filas.")
        return

    stored = np.load(ids_path, allow_pickle=True)
    if stored.shape[0] != n_exp:
        raise AssertionError(
            f"{repr_name}: hay {stored.shape[0]} ids guardados pero {n_exp} filas."
        )

    if not np.array_equal(stored, expected):
        raise AssertionError(
            f"{repr_name}: los tupla_id de la matriz no coinciden con los del dataset."
        )

In [8]:
def cargar_matriz_global(repr_name, expected_ids=None):
    '''Carga los embeddings de FinBERT o SBERT. TF-IDF devuelve None (se hace por fold).'''
    if repr_name == "tfidf":
        return None
    matriz_path = MATRIX_PATHS[repr_name]
    matriz = np.load(matriz_path)
    if expected_ids is not None:
        _verificar_alineacion_matriz(matriz_path, matriz, expected_ids, repr_name)
    return matriz

### DBCV

DBCV (*Density-Based Clustering Validation*) es la métrica natural para HDBSCAN porque está pensada para clústeres de densidad y formas arbitrarias. El problema es que `validity_index` no está definido cuando la partición deja menos de dos clústeres no-ruido. Devolvemos `NaN` en ese caso para  no detener toda la búsqueda.

In [9]:
def calcular_dbcv(embeddings, labels_pred):
    # validity_index revienta si queda menos de 2 clusters de verdad asi que ponemos NaN.
    mask = labels_pred != -1
    if len(set(labels_pred[mask])) < 2:
        return np.nan
    try:
        return float(validity_index(
            embeddings.astype(np.float64), labels_pred, metric="euclidean"))
    except Exception:
        return np.nan

### Métricas de un fold

Distinguimos dos familias de métricas según el periodo sobre el que se calculan. Las **globales** —Silhoutette Score y DBCV— miden la calidad geométrica de la partición sobre toda la historia $\le T$: cuán separados y densos son los clústeres en el espacio proyectado. Las **locales** se restringen a las señales del año $T$ y describen la partición tal y como se observa en el momento de la evaluación: número de clústeres vivos, proporción de ruido y NMI de las etiquetas frente al sector GICS. Contamos los clústeres localmente, sobre el año $T$, porque es la partición que de hecho usaríamos para predecir, no la del histórico completo.

In [10]:
def evaluar_clusters(emb_train, labels_train, idx_T_local, gics_T):
    '''Metricas globales (toda la historia <= T) y locales (solo año T).'''
    # --- Globales: calidad geometrica sobre toda la historia <= T ---
    mask_no_ruido_global = labels_train != -1
    labels_in_global = labels_train[mask_no_ruido_global]

    if len(set(labels_in_global)) < 2:
        dbcv_global, sil_global = np.nan, np.nan
    else:
        sil_global = float(silhouette_score(
            emb_train[mask_no_ruido_global], labels_in_global,
            metric=SILHOUETTE_METRIC,
        ))
        dbcv_global = calcular_dbcv(emb_train, labels_train)

    # --- Locales: solo las señales del año T ---
    labels_T = labels_train[idx_T_local]
    mask_no_ruido_T = labels_T != -1

    labels_in_T = labels_T[mask_no_ruido_T]
    n_clusters_T = len(set(labels_in_T))

    if len(labels_T) == 0:
        noise_ratio_T = float("nan")
    else:
        noise_ratio_T = 1.0 - float(mask_no_ruido_T.mean())

    # NMI contra GICS: solo tiene sentido donde hay etiqueta de sector y no es ruido.
    gics_arr = gics_T.to_numpy()
    mask_externa_T = mask_no_ruido_T & (~pd.isna(gics_arr))

    if mask_externa_T.sum() < 2 or len(set(labels_T[mask_externa_T])) < 2:
        nmi_T = float("nan")
    else:
        nmi_T = float(normalized_mutual_info_score(
            gics_arr[mask_externa_T], labels_T[mask_externa_T],
        ))

    return {
        "n_clusters":  int(n_clusters_T),
        "noise_ratio": float(noise_ratio_T),
        "dbcv":        float(dbcv_global) if not np.isnan(dbcv_global) else np.nan,
        "silhouette":  float(sil_global),
        "nmi":         float(nmi_T) if not np.isnan(nmi_T) else np.nan,
    }

## Bloque 3 — Generación de folds

Cargamos el dataset lematizado y construimos las ventanas temporales, imprimiéndolas antes de lanzar nada pesado para confirmar que la partición train/evaluación es la que esperamos.

In [11]:
df_nlp = pd.read_parquet(DATASET_LEMATIZADO)
print(f"Dataset lematizado: {len(df_nlp)} documentos, {df_nlp['ticker'].nunique()} tickers")
print(f"Rango temporal: {df_nlp['quarter'].min()} -> {df_nlp['quarter'].max()}")
df_nlp.head(3)

Dataset lematizado: 8893 documentos, 454 tickers
Rango temporal: 2019Q1 -> 2023Q4


,tupla_id,ticker,company,sector,earnings_date,quarter,presentation,has_qa,texto_valido,palabras_presentacion,presentation_limpia
0,LEN_2019Q1,LEN,Lennar,Consumer Discretionary,2019-01-09,2019Q1,Alexandra Lumpkin for the reading of the forwa...,False,1,9262,alexandra lumpkin reading forward look stateme...
1,STZ_2019Q1,STZ,Constellation Brands,Consumer Staples,2019-01-09,2019Q1,"Patty Yahn-Urlaub, Senior Vice President of In...",True,1,4072,patty yahn urlaub senior vice president invest...
2,C_2019Q1,C,Citigroup,Financials,2019-01-14,2019Q1,﻿ Operator : Hello. And welcome to Citi’s Four...,True,1,5179,hello welcome citi fourth earnings review toda...


In [12]:
ventanas = generar_ventanas_walk_forward(df_nlp["quarter"].astype(str).unique())
N_FOLDS = len(ventanas)  # nº de ventanas temporales; base del filtro de estabilidad

print(f"{N_FOLDS} folds walk-forward:\n")
for v in ventanas:
    print(f"  Fold {v['fold']}: ajuste hasta {v['train_end_year']} "
          f"({len(v['train_quarters'])} trimestres) -> metricas sobre {v['train_end_year']}")


3 folds walk-forward:

  Fold 1: ajuste hasta 2020 (8 trimestres) -> metricas sobre 2020
  Fold 2: ajuste hasta 2021 (12 trimestres) -> metricas sobre 2021
  Fold 3: ajuste hasta 2022 (16 trimestres) -> metricas sobre 2022


## Bloque 4 — Proyecciones UMAP por fold

UMAP es, con diferencia, la parte más costosa del cálculo. Para una representación y un año $T$ fijos, la proyección no depende de los hiperparámetros de HDBSCAN, así que la calculamos una sola vez por par $(\text{representación}, \text{fold})$ y la reutilizamos en toda la búsqueda en rejilla del bloque siguiente. Esto evita repetir cientos de veces la misma reducción de dimensión.

Reducimos a 50 dimensiones antes de clusterizar porque sobre los embeddings densos originales HDBSCAN se vuelve muy inestable: en alta dimensión las distancias se concentran y la noción de densidad pierde poder discriminante. UMAP preserva la estructura local de vecindad y deja un espacio donde el clustering por densidad se comporta mucho mejor.

Para TF-IDF ajustamos el vectorizador dentro del fold, solo con los textos del histórico $\le T$, de modo que el vocabulario no incorpore términos que solo aparecen en años futuros. Para FinBERT y SBERT, que son embeddings preentrenados y no se reajustan sobre nuestro corpus, basta con indexar la matriz global por las filas del train.

In [13]:
warnings.filterwarnings('ignore')

# Vector de tupla_id en el orden del dataset, para verificar la alineacion de
# FinBERT/SBERT por id y no solo por numero de filas.
expected_ids = df_nlp["tupla_id"].values

# Año de cada documento a partir de los 4 primeros caracteres del trimestre.
quarter_to_year = df_nlp["quarter"].astype(str).str[:4].astype(int)

cache_umap = {}

for repr_config in REPRESENTACIONES:
    repr_name = repr_config["name"]


    matriz_global = cargar_matriz_global(repr_name, expected_ids=expected_ids)

    for ventana in ventanas:
        year_T = ventana["train_end_year"]

        # Histórico <= T y, dentro de el, la posicion local de las señales del año T.
        mask_train = quarter_to_year <= year_T
        idx_train = np.flatnonzero(mask_train.to_numpy())
        if idx_train.size == 0:
            continue

        mask_T_on_train = (quarter_to_year.to_numpy()[idx_train] == year_T)
        idx_T_local = np.flatnonzero(mask_T_on_train)

        # TF-IDF: se ajusta solo con el historico <= T para no colar vocabulario futuro.
        if repr_name == "tfidf":
            textos_train = df_nlp.iloc[idx_train]["presentation_limpia"].astype(str).tolist()
            vectorizer = TfidfVectorizer(
                max_df=TFIDF_MAX_DF,
                min_df=TFIDF_MIN_DF,
                max_features=TFIDF_MAX_FEATURES,
                ngram_range=(1, 2),
                sublinear_tf=True,
            )
            mat_train = vectorizer.fit_transform(textos_train)
            if issparse(mat_train):
                mat_train = mat_train.toarray()
        else:
            mat_train = matriz_global[idx_train]

        # UMAP sobre el historico ya aislado.
        reducer = umap.UMAP(
            n_neighbors=UMAP_PARAMS["n_neighbors"],
            n_components=UMAP_PARAMS["n_components"],
            metric=UMAP_PARAMS["metric"],
            random_state=UMAP_PARAMS["random_state"],
        )
        emb_train = reducer.fit_transform(mat_train)

        df_T = (df_nlp.iloc[idx_train].iloc[idx_T_local]
                [["ticker", "quarter", "sector"]].reset_index(drop=True))

        cache_umap[(repr_name, year_T)] = {
            "emb_train":   emb_train,
            "idx_T_local": idx_T_local,
            "df_T":        df_T,
            "n_train":     int(len(idx_train)),
            "n_signals_T": int(len(idx_T_local)),
        }
        print(f"[{repr_name}] Año {year_T} | Histórico: {len(idx_train)} docs | Señales nuevas: {len(idx_T_local)}")

[tfidf] Año 2020 | Histórico: 3553 docs | Señales nuevas: 1784
[tfidf] Año 2021 | Histórico: 5322 docs | Señales nuevas: 1769
[tfidf] Año 2022 | Histórico: 7105 docs | Señales nuevas: 1783
[finbert] Año 2020 | Histórico: 3553 docs | Señales nuevas: 1784
[finbert] Año 2021 | Histórico: 5322 docs | Señales nuevas: 1769
[finbert] Año 2022 | Histórico: 7105 docs | Señales nuevas: 1783
[sbert] Año 2020 | Histórico: 3553 docs | Señales nuevas: 1784
[sbert] Año 2021 | Histórico: 5322 docs | Señales nuevas: 1769
[sbert] Año 2022 | Histórico: 7105 docs | Señales nuevas: 1783


## Bloque 5 — Búsqueda en rejilla de HDBSCAN

Recorremos las 192 combinaciones de hiperparámetros de cada representación. Para cada combinación evaluamos HDBSCAN sobre todos los folds, reutilizando la proyección UMAP guardada, y resumimos las métricas promediando a través de los folds. Trabajar con la media entre folds reduce el efecto de un año atípico y nos da una estimación más estable de cómo se comporta cada configuración a lo largo del tiempo, en lugar de premiar la que acierta en un único periodo.

In [14]:
# Generamos las 192 combinaciones. Ordenamos las claves para que el orden sea siempre el mismo entre ejecuciones.
def generar_combinaciones(grid):
    keys = sorted(grid.keys())
    values = [grid[k] for k in keys]
    for combo in itertools.product(*values):
        yield dict(zip(keys, combo))

combinaciones = list(generar_combinaciones(HDBSCAN_GRID))
print(f"{len(combinaciones)} combinaciones por representacion "
      f"({len(combinaciones) * len(REPRESENTACIONES)} experimentos en total)")

192 combinaciones por representacion (576 experimentos en total)


In [15]:
def agregar_metricas(metricas_por_fold):
    '''Media, desviacion (ddof=1) y nº de folds validos de cada metrica.'''
    agregado = {}
    for k in METRIC_KEYS:
        vals = np.array(
            [m[k] for m in metricas_por_fold if m.get(k) is not None],
            dtype=float,
        )
        vals = vals[~np.isnan(vals)]
        if vals.size == 0:
            agregado[k] = np.nan
            agregado[f"{k}_std"] = np.nan
            agregado[f"{k}_nfolds"] = 0
        else:
            agregado[k] = round(float(np.mean(vals)), 4)
            agregado[f"{k}_std"] = round(float(np.std(vals, ddof=1)), 4) if vals.size > 1 else 0.0
            agregado[f"{k}_nfolds"] = int(vals.size)
    return agregado

In [16]:
resultados = []

for repr_config in REPRESENTACIONES:
    repr_name = repr_config["name"]
    print(f"\n{'=' * 60}\n  {repr_name.upper()}\n{'=' * 60}")

    for i, combo in enumerate(combinaciones, 1):
        experiment_id = (
            f"{repr_name}_mcs{combo['min_cluster_size']}_ms{combo['min_samples']}"
            f"_eps{combo['cluster_selection_epsilon']}_{combo['cluster_selection_method']}"
        )

        metricas_por_fold = []
        for ventana in ventanas:
            year_T = ventana["train_end_year"]
            entry = cache_umap.get((repr_name, year_T))
            if entry is None:
                continue

            emb_train = entry["emb_train"]
            idx_T_local = entry["idx_T_local"]
            df_T = entry["df_T"]

            clusterer = hdbscan.HDBSCAN(
                min_cluster_size=combo["min_cluster_size"],
                min_samples=combo["min_samples"],
                cluster_selection_epsilon=combo["cluster_selection_epsilon"],
                cluster_selection_method=combo["cluster_selection_method"],
                metric="euclidean",
            )
            labels_train = clusterer.fit_predict(emb_train)

            m = evaluar_clusters(emb_train, labels_train, idx_T_local, df_T["sector"])
            metricas_por_fold.append(m)

        if not metricas_por_fold:
            agregado = {k: np.nan for k in METRIC_KEYS}
            agregado.update({f"{k}_std": np.nan for k in METRIC_KEYS})
            agregado.update({f"{k}_nfolds": 0 for k in METRIC_KEYS})
        else:
            agregado = agregar_metricas(metricas_por_fold)

        resultados.append({
            "experiment_id": experiment_id,
            "representation": repr_name,
            **combo,
            **agregado,
        })

        if i % 40 == 0 or i == len(combinaciones):
            print(f"  [{repr_name}] {i}/{len(combinaciones)} combinaciones")


  TFIDF
  [tfidf] 40/192 combinaciones
  [tfidf] 80/192 combinaciones
  [tfidf] 120/192 combinaciones
  [tfidf] 160/192 combinaciones
  [tfidf] 192/192 combinaciones

  FINBERT
  [finbert] 40/192 combinaciones
  [finbert] 80/192 combinaciones
  [finbert] 120/192 combinaciones
  [finbert] 160/192 combinaciones
  [finbert] 192/192 combinaciones

  SBERT
  [sbert] 40/192 combinaciones
  [sbert] 80/192 combinaciones
  [sbert] 120/192 combinaciones
  [sbert] 160/192 combinaciones
  [sbert] 192/192 combinaciones


Reunimos todos los experimentos en un DataFrame, ordenamos las columnas y lo guardamos en `registry.csv`

In [17]:
df_registry = pd.DataFrame(resultados)

cols_orden = (
    ["experiment_id", "representation", "min_cluster_size", "min_samples",
     "cluster_selection_epsilon", "cluster_selection_method"]
    + list(METRIC_KEYS)
    + [f"{k}_std" for k in METRIC_KEYS]
    + [f"{k}_nfolds" for k in METRIC_KEYS]
)
df_registry = df_registry[[c for c in cols_orden if c in df_registry.columns]]

df_registry.to_csv(REGISTRY_PATH, index=False)
print(f"registry.csv guardado: {len(df_registry)} experimentos -> {REGISTRY_PATH}")
df_registry.head()

registry.csv guardado: 576 experimentos -> C:\Users\diego\TFG\experiments\registry.csv


,experiment_id,representation,min_cluster_size,min_samples,cluster_selection_epsilon,cluster_selection_method,dbcv,silhouette,nmi,n_clusters,...,dbcv_std,silhouette_std,nmi_std,n_clusters_std,noise_ratio_std,dbcv_nfolds,silhouette_nfolds,nmi_nfolds,n_clusters_nfolds,noise_ratio_nfolds
0,tfidf_mcs150_ms5_eps0.0_eom,tfidf,150,5,0.0,eom,0.2287,0.5904,0.7259,13.6667,...,0.0881,0.0570,0.0591,3.5119,0.0303,3,3,3,3,3
1,tfidf_mcs150_ms15_eps0.0_eom,tfidf,150,15,0.0,eom,0.2424,0.5993,0.7304,13.3333,...,0.0674,0.0490,0.0629,3.5119,0.0538,3,3,3,3,3
2,tfidf_mcs150_ms20_eps0.0_eom,tfidf,150,20,0.0,eom,0.2216,0.5937,0.7250,13.0000,...,0.0595,0.0497,0.0618,4.0000,0.0774,3,3,3,3,3
3,tfidf_mcs150_ms30_eps0.0_eom,tfidf,150,30,0.0,eom,0.2512,0.5962,0.7272,13.3333,...,0.0751,0.0429,0.0559,4.0415,0.0552,3,3,3,3,3
4,tfidf_mcs200_ms5_eps0.0_eom,tfidf,200,5,0.0,eom,0.1733,0.5002,0.6612,9.3333,...,0.0830,0.1388,0.1430,4.7258,0.0181,3,3,3,3,3


## Bloque 6 — Selección de la Fase 1

De los 576 experimentos nos quedamos con un ganador por representación. Aplicamos dentro de cada familia una cadena de etapas:

1. **Viabilidad estructural y estabilidad inter-ventanas:** descartamos los experimentos con DBCV o Silhouette no definidos y exigimos `noise_ratio < 0.50`. Además, una configuración solo es viable si produce métricas geométricas válidas (al menos dos clústeres no triviales) en **todas** las ventanas temporales, no solo en algunas. Esto descarta parametrizaciones que colapsan en algún régimen de mercado y cuya media se calcularía sobre un único año, evitando que compitan en igualdad con configuraciones estables en todo el periodo.
2. **Calidad relativa dentro de la familia:** conservamos los que quedan en la mediana o por encima, tanto en DBCV como en Silhouette.
3. **Orden lexicográfico:** DBCV descendente, luego Silhouette descendente, y como último desempate `noise_ratio` ascendente. El orden de prioridad refleja que DBCV es la métrica más fiel a la estructura de densidad que buscamos.
4. **Top-1** de cada representación.


In [18]:
# Umbral de ruido y orden de las representaciones.
FILTRO_RUIDO_MAX = 0.50
LISTA_REPR = ["tfidf", "finbert", "sbert"]

# Releemos registry.csv para dejar claro que la seleccion solo depende de ese fichero.
df_all = pd.read_csv(REGISTRY_PATH)
print(f"Registro: {len(df_all)} experimentos")
for r in LISTA_REPR:
    print(f"  {r}: {(df_all['representation'] == r).sum()} experimentos")

Registro: 576 experimentos
  tfidf: 192 experimentos
  finbert: 192 experimentos
  sbert: 192 experimentos


Recorremos las representaciones una a una, aplicando las cuatro etapas y guardando el ganador de cada familia.

In [19]:
campeones = []

for repr_name in LISTA_REPR:
    df_repr = df_all[df_all["representation"] == repr_name].copy()
    print(f"\n{'-' * 60}\n[{repr_name.upper()}] {len(df_repr)} experimentos\n{'-' * 60}")

    # Etapa 1a: fuera DBCV/silueta no definidos y ruido excesivo.
    df_e1 = df_repr.dropna(subset=["dbcv", "silhouette"]).copy()
    df_e1 = df_e1[df_e1["noise_ratio"] < FILTRO_RUIDO_MAX]

    # Etapa 1b: estabilidad inter-ventanas. Una configuracion solo es viable si
    # arroja metricas validas en TODAS las ventanas (N_FOLDS), no solo en algunas.
    # Asi evitamos que una media calculada sobre 1 fold compita con medias sobre todos.
    cols_estab = ["dbcv_nfolds", "silhouette_nfolds", "noise_ratio_nfolds"]
    cols_estab = [c for c in cols_estab if c in df_e1.columns]
    for c in cols_estab:
        df_e1 = df_e1[df_e1[c] == N_FOLDS]
    print(f"  Etapa 1 (ruido < {FILTRO_RUIDO_MAX}; dbcv/sil no nulos; "
          f"validos en {N_FOLDS}/{N_FOLDS} ventanas): {len(df_repr)} -> {len(df_e1)}")
    if df_e1.empty:
        print(f"  AVISO: {repr_name} sin supervivientes tras Etapa 1.")
        continue

    # Etapa 2: nos quedamos en la mediana o por encima en ambas metricas.
    mediana_dbcv = float(df_e1["dbcv"].median())
    mediana_sil  = float(df_e1["silhouette"].median())
    df_e2 = df_e1[
        (df_e1["dbcv"] >= mediana_dbcv)
        & (df_e1["silhouette"] >= mediana_sil)
    ].copy()
    print(f"  Etapa 2 (dbcv >= {mediana_dbcv:.4f} y sil >= {mediana_sil:.4f}): "
          f"{len(df_e1)} -> {len(df_e2)}")
    if df_e2.empty:
        print(f"  AVISO: {repr_name} sin supervivientes tras Etapa 2.")
        continue

    # Etapa 3: orden lexicografico DBCV -> silueta -> ruido.
    df_ranked = df_e2.sort_values(
        by=["dbcv", "silhouette", "noise_ratio"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    # Etapa 4: el mejor de la familia.
    ganador = df_ranked.head(1).copy()
    ganador["rank_within_repr"] = 1
    campeones.append(ganador)

    fila = ganador.iloc[0]
    print(f"  Ganador: {fila['experiment_id']}")
    print(f"    DBCV={fila['dbcv']:.4f} | Sil={fila['silhouette']:.4f} | "
          f"Ruido={fila['noise_ratio']:.2%} | Clusters~{fila['n_clusters']:.2f}")



------------------------------------------------------------
[TFIDF] 192 experimentos
------------------------------------------------------------
  Etapa 1 (ruido < 0.5; dbcv/sil no nulos; validos en 3/3 ventanas): 192 -> 176
  Etapa 2 (dbcv >= 0.1315 y sil >= 0.4629): 176 -> 76
  Ganador: tfidf_mcs150_ms30_eps0.0_eom
    DBCV=0.2512 | Sil=0.5962 | Ruido=29.54% | Clusters~13.33

------------------------------------------------------------
[FINBERT] 192 experimentos
------------------------------------------------------------
  Etapa 1 (ruido < 0.5; dbcv/sil no nulos; validos en 3/3 ventanas): 192 -> 185
  Etapa 2 (dbcv >= 0.1866 y sil >= 0.4398): 185 -> 43
  Ganador: finbert_mcs400_ms15_eps0.0_leaf
    DBCV=0.3124 | Sil=0.5037 | Ruido=32.93% | Clusters~4.33

------------------------------------------------------------
[SBERT] 192 experimentos
------------------------------------------------------------
  Etapa 1 (ruido < 0.5; dbcv/sil no nulos; validos en 3/3 ventanas): 192 -> 124
  

Juntamos los tres campeones y les damos un rango global mediante los mismo criterios.

In [20]:
df_finalistas = pd.concat(campeones, ignore_index=True)

df_finalistas = df_finalistas.sort_values(
    by=["dbcv", "silhouette", "noise_ratio"],
    ascending=[False, False, True],
).reset_index(drop=True)
df_finalistas["rank"] = range(1, len(df_finalistas) + 1)

print(f"{len(df_finalistas)} finalistas (1 por representacion)")
print("Orden global (DBCV desc -> Sil desc -> ruido asc):\n")
for _, row in df_finalistas.iterrows():
    print(f"  #{int(row['rank'])}: {row['experiment_id']} [{row['representation']}]")
    print(f"      DBCV={row['dbcv']:.4f} | Sil={row['silhouette']:.4f} | "
          f"Ruido={row['noise_ratio']:.2%}")

df_finalistas

3 finalistas (1 por representacion)
Orden global (DBCV desc -> Sil desc -> ruido asc):

  #1: finbert_mcs400_ms15_eps0.0_leaf [finbert]
      DBCV=0.3124 | Sil=0.5037 | Ruido=32.93%
  #2: tfidf_mcs150_ms30_eps0.0_eom [tfidf]
      DBCV=0.2512 | Sil=0.5962 | Ruido=29.54%
  #3: sbert_mcs150_ms30_eps0.0_leaf [sbert]
      DBCV=0.2233 | Sil=0.6386 | Ruido=33.79%


,experiment_id,representation,min_cluster_size,min_samples,cluster_selection_epsilon,cluster_selection_method,dbcv,silhouette,nmi,n_clusters,...,nmi_std,n_clusters_std,noise_ratio_std,dbcv_nfolds,silhouette_nfolds,nmi_nfolds,n_clusters_nfolds,noise_ratio_nfolds,rank_within_repr,rank
0,finbert_mcs400_ms15_eps0.0_leaf,finbert,400,15,0.0,leaf,0.3124,0.5037,0.2952,4.3333,...,0.2482,2.0817,0.2541,3,3,3,3,3,1,1
1,tfidf_mcs150_ms30_eps0.0_eom,tfidf,150,30,0.0,eom,0.2512,0.5962,0.7272,13.3333,...,0.0559,4.0415,0.0552,3,3,3,3,3,1,2
2,sbert_mcs150_ms30_eps0.0_leaf,sbert,150,30,0.0,leaf,0.2233,0.6386,0.7496,10.6667,...,0.0411,3.0551,0.0441,3,3,3,3,3,1,3


Guardamos los tres candidatos en `Seleccion_Finalistas.csv`. Con sus hiperparámetros de HDBSCAN ya congelados, son la entrada de la validación económica de la Fase 2.

In [21]:
df_finalistas.to_csv(FINALISTAS_PATH, index=False)
print(f"Finalistas guardados -> {FINALISTAS_PATH}")
df_finalistas[["rank", "representation", "experiment_id",
               "min_cluster_size", "min_samples",
               "cluster_selection_epsilon", "cluster_selection_method",
               "dbcv", "silhouette", "noise_ratio", "n_clusters"]]

Finalistas guardados -> C:\Users\diego\TFG\experiments\Seleccion_Finalistas.csv


,rank,representation,experiment_id,min_cluster_size,min_samples,cluster_selection_epsilon,cluster_selection_method,dbcv,silhouette,noise_ratio,n_clusters
0,1,finbert,finbert_mcs400_ms15_eps0.0_leaf,400,15,0.0,leaf,0.3124,0.5037,0.3293,4.3333
1,2,tfidf,tfidf_mcs150_ms30_eps0.0_eom,150,30,0.0,eom,0.2512,0.5962,0.2954,13.3333
2,3,sbert,sbert_mcs150_ms30_eps0.0_leaf,150,30,0.0,leaf,0.2233,0.6386,0.3379,10.6667
